<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Multi-Agent Patterns
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M19-Multi-Agent-Patterns"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

**Die LangGraph-Module** zeigten einzelne Graphen mit bedingtem Routing, Checkpointing und HITL. **Dieses Modul** geht einen Schritt weiter: mehrere spezialisierte Agenten arbeiten zusammen.



**Warum reicht ein einzelner Agent oft nicht?**

| Einzel-Agent | Multi-Agent |
|-------------|-------------|
| Allgemein, kein Fokuswissen | Spezialist pro Aufgabenbereich |
| Sequentiell, nicht skalierbar | Parallel oder koordiniert ausführbar |
| Eine große Aufgabe überfordert LLM | Aufgaben aufgeteilt in überschaubare Schritte |
| Schwer debuggbar | Jeder Agent einzeln testbar |



**Drei Haupt-Patterns in LangGraph:**

| Pattern | Koordination | Stärken | Typischer Einsatz |
|---------|-------------|---------|------------------|
| **Supervisor** | LLM entscheidet dynamisch | Flexibel, selbstorganisiert | Offene Aufgaben |
| **Hierarchisch** | Subgraphen als Nodes | Modular, wiederverwendbar | Komplexe Workflows |
| **Pipeline** | Feste Reihenfolge | Einfach, deterministisch | ETL, Content-Erzeugung |

> **Gemeinsame Basis:** Shared State – alle Agenten lesen und schreiben  
> denselben `TypedDict`-State, koordiniert über `add_messages`.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Drei Multi-Agent Pattern</font> </br></p>

diagram = '''
flowchart LR
    subgraph SUP["🟢 Supervisor"]
        direction TB
        SV["🎯 Supervisor\nLLM-Routing"]
        SV --> SA["Agent 1"]
        SV --> SB["Agent 2"]
        SV --> SC["Agent 3"]
        SA & SB & SC -->|"fertig"| SV
    end

    subgraph HIE["🔵 Hierarchisch"]
        direction TB
        TS["Haupt-Graph"]
        TS --> SS1["📦 Sub-Graph 1"]
        TS --> SS2["📦 Sub-Graph 2"]
    end

    subgraph PIP["🟡 Pipeline"]
        direction LR
        P1["Agent 1"] -->|"State"| P2["Agent 2"] -->|"State"| P3["Agent 3"]
    end
'''

mermaid(diagram, width=980)

# 2 | Warum Multi-Agent?
---

**Spezialisierung** ist der zentrale Vorteil:  
Jeder Agent erhält einen fokussierten System-Prompt – das verbessert Qualität sprunghaft.

```python
# Einzel-Agent: macht alles, aber keines richtig gut
llm.invoke([SystemMessage("Du bist Assistent."), HumanMessage(aufgabe)])

# Multi-Agent: jeder Spezialist in seinem Bereich
llm.invoke([SystemMessage("Du bist Recherche-Spezialist. Fakten, keine Wertung."), ...])
llm.invoke([SystemMessage("Du bist Analyst. Strukturiere und bewerte Informationen."), ...])
llm.invoke([SystemMessage("Du bist Redakteur. Schreibe klar und prägnant."), ...])
```

**Weitere Vorteile:**

- 🔄 **Wiederholbarkeit:** Einzelne Agenten können neu gestartet werden
- 🧪 **Testbarkeit:** Jeder Agent separat testbar und optimierbar
- 📏 **Kontext-Management:** Jeder Agent hat nur relevante Nachrichten in seinem Context
- 🔧 **Modularität:** Agenten austauschbar ohne den Gesamtworkflow zu ändern

> **Faustregel:** Multi-Agent lohnt sich, wenn eine Aufgabe klar in **2+ unabhängige Phasen**  
> mit unterschiedlichen Anforderungen aufgeteilt werden kann.

# 3 | Pattern 1: Supervisor
---

Der **Supervisor** ist ein LLM, das den Workflow dynamisch steuert.  
Nach jedem Agenten-Beitrag entscheidet er: Wer arbeitet als nächstes? Oder: Fertig?

**Schlüssel-API:** `with_structured_output()`

```python
from pydantic import BaseModel, Field
from typing import Literal

class SupervisorEntscheidung(BaseModel):
    naechster: Literal["recherche", "analyse", "schreiben", "FINISH"]

supervisor_llm = llm.with_structured_output(SupervisorEntscheidung)
```

**Topologie:** Sternförmig – alle Agenten melden sich nach ihrer Arbeit beim Supervisor zurück.

**State-Feld `naechster`:**  
Der Supervisor schreibt den nächsten Agent-Namen in dieses Feld.  
Die Router-Funktion liest es und leitet den Graphen weiter.

> **📚 Modell-Auswahl: Supervisor vs. Worker**
>
> Der Supervisor trifft Routing-Entscheidungen → starkes Reasoning-Modell (`o3`).
> Worker generieren Inhalte → schnelles, kosteneffizientes Modell (`gpt-4o-mini`).
> ⚠️ `o3` und `o3-mini` akzeptieren **keinen** `temperature`-Parameter.
>
> Vollständige Regeln: [Modell-Auswahl Guide](https://ralf-42.github.io/Agenten/frameworks/Modell_Auswahl_Guide.html)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Supervisor-Pattern Detail</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nwith\_structured\_output()"]

    SUP -->|"recherche"| R["🔍 Recherche-Agent\nFakten sammeln"]
    SUP -->|"analyse"| A["📊 Analyse-Agent\nBewerten & Strukturieren"]
    SUP -->|"schreiben"| W["✍️ Schreib-Agent\nBericht formulieren"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage(name='Recherche')"| SUP
    A -->|"AIMessage(name='Analyse')"| SUP
    W -->|"AIMessage(name='Schreiben')"| SUP

    style SUP fill:#FF9800,color:#fff
    style R   fill:#2196F3,color:#fff
    style A   fill:#9C27B0,color:#fff
    style W   fill:#4CAF50,color:#fff
'''

mermaid(diagram, width=750)

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image as IPImage

# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES = 3   # API-Retry-Versuche bei transienten Fehlern (with_retry)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# ── State ──────────────────────────────────────────────────────────────────
class SupervisorState(TypedDict):
    messages:  Annotated[list, add_messages]
    aufgabe:   str   # Original-Aufgabe
    naechster: str   # Routing-Feld: wer kommt als nächstes

# ── Strukturierter Supervisor-Output ───────────────────────────────────────
class SupervisorEntscheidung(BaseModel):
    """Routing-Entscheidung des Supervisors."""
    naechster: Literal["recherche", "analyse", "schreiben", "FINISH"] = Field(
        description="Nächster Agent oder FINISH wenn fertig."
    )

# o3 akzeptiert keinen temperature-Parameter (API-Fehler bei temperature=0.0)
# .with_retry(): schützt vor transienten API-Fehlern beim Routing
supervisor_llm = (
    init_chat_model("openai:o3")
    .with_structured_output(SupervisorEntscheidung)
    .with_retry(stop_after_attempt=MAX_RETRIES)
)

SUPERVISOR_SYSTEM = SystemMessage(content=(
    "Du bist Supervisor eines Analyse-Teams.\n\n"
    "Dein Team:\n"
    "- recherche: Sammelt Fakten (zuerst aufrufen)\n"
    "- analyse: Bewertet die Fakten (nach Recherche)\n"
    "- schreiben: Formuliert den Bericht (am Ende)\n\n"
    "Reihenfolge: recherche → analyse → schreiben → FINISH"
))

# ── Spezialisierte Agenten ─────────────────────────────────────────────────
def recherche_agent(state: SupervisorState) -> dict:
    """Sammelt 3 prägnante Fakten zum Thema."""
    resp = llm.invoke([
        SystemMessage("Du bist Recherche-Spezialist. "
                      "Liefere genau 3 Stichpunkte mit Fakten. Kurz und präzise. Deutsch."),
        HumanMessage(f"Thema: {state['aufgabe']}"),
    ])
    return {"messages": [AIMessage(content=resp.content, name="Recherche")]}

def analyse_agent(state: SupervisorState) -> dict:
    """Wertet die Recherche-Ergebnisse aus."""
    resp = llm.invoke([
        SystemMessage("Du bist Analyse-Spezialist. Bewerte die Informationen: "
                      "Stärken, Schwächen, Fazit. Je 1 Satz. Deutsch."),
        *state["messages"],
    ])
    return {"messages": [AIMessage(content=resp.content, name="Analyse")]}

def schreiben_agent(state: SupervisorState) -> dict:
    """Schreibt den finalen Bericht."""
    resp = llm.invoke([
        SystemMessage("Du bist Schreib-Spezialist. Erstelle einen strukturierten Bericht "
                      "(Einleitung + 3 Punkte + Fazit). Max. 120 Wörter. Deutsch."),
        *state["messages"],
    ])
    return {"messages": [AIMessage(content=resp.content, name="Schreiben")]}

def supervisor_node(state: SupervisorState) -> dict:
    """LLM entscheidet, welcher Agent als nächstes aktiv sein soll."""
    entscheidung = supervisor_llm.invoke([SUPERVISOR_SYSTEM] + state["messages"])
    return {"naechster": entscheidung.naechster}

def supervisor_router(state: SupervisorState) -> str:
    naechster = state.get("naechster", "FINISH")
    return "__end__" if naechster == "FINISH" else naechster

# ── Graph ──────────────────────────────────────────────────────────────────
builder_s = StateGraph(SupervisorState)
builder_s.add_node("supervisor", supervisor_node)
builder_s.add_node("recherche",  recherche_agent)
builder_s.add_node("analyse",    analyse_agent)
builder_s.add_node("schreiben",  schreiben_agent)

builder_s.add_edge(START, "supervisor")
builder_s.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {"recherche": "recherche", "analyse": "analyse",
     "schreiben": "schreiben", "__end__": END}
)
# Jeder Agent meldet sich nach der Arbeit beim Supervisor zurück
builder_s.add_edge("recherche", "supervisor")
builder_s.add_edge("analyse",   "supervisor")
builder_s.add_edge("schreiben", "supervisor")

supervisor_graph = builder_s.compile()

In [ ]:
display(IPImage(supervisor_graph.get_graph().draw_mermaid_png()))

In [ ]:
run_cfg = {"run_name": "M19-Kap3-Supervisor", "tags": ["m19", "supervisor"]}

result = supervisor_graph.invoke(
    {"messages": [],
     "aufgabe":   "Vor- und Nachteile von Multi-Agent-Systemen in der Softwareentwicklung",
     "naechster": ""},
    config=run_cfg
)

print(f"Nachrichten gesamt: {len(result['messages'])}")
print()
for msg in result["messages"]:
    agent_name = getattr(msg, 'name', None) or msg.__class__.__name__
    mprint(f"**[{agent_name}]**\n{msg.content}\n---")

# 4 | Pattern 2: Subgraphen &amp; Hierarchie
---

Beim **Hierarchischen Pattern** werden komplette Graphen als Nodes  
in einem übergeordneten Graphen eingebettet – **Subgraphen**.

```python
# Subgraph kompilieren
sub_graph = sub_builder.compile()

# Als Node im Hauptgraph verwenden
haupt_builder.add_node("mein_team", sub_graph)
haupt_builder.add_edge(START, "mein_team")
haupt_builder.add_edge("mein_team", "naechster_node")
```

**Voraussetzung:** Der Subgraph muss **kompatible State-Felder** mit dem Hauptgraph teilen.  
LangGraph übergibt automatisch die passenden Felder.

**Wann Subgraphen verwenden?**

| Szenario | Beschreibung |
|----------|-------------|
| **Wiederverwendung** | Ein Recherche-Team wird in mehreren Workflows genutzt |
| **Kapselung** | Interne Logik des Sub-Teams verbirgt sich hinter einer Schnittstelle |
| **Testen** | Subgraph isoliert testbar; Hauptgraph separat |
| **Skalierung** | Sub-Teams können parallel ausgeführt werden |

> **`get_graph(xray=True)`** zeigt die vollständige Hierarchie inkl. Subgraph-Internals.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Hierarchisches Pattern Detail</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> HG["Manager / Orchestrator"]

    subgraph IBOX_INT["📦 Informations-Team"]
        direction LR
        IS["📡 Sammler"] -->|"roh_info"| IF["🔎 Faktenprüfer"]
    end

    HG --> IS
    IF --> BS["✍️ Bericht-Schreiber"]
    BS --> END([END])

    %% Styling
    style IBOX_INT fill:#f1f7ff,stroke:#2196F3,stroke-width:2px
    style IS fill:#2196F3,color:#fff
    style IF fill:#2196F3,color:#fff
    style BS fill:#4CAF50,color:#fff
    style HG fill:#FF9800,color:#fff
'''

mermaid(diagram, width=780)

In [ ]:
# ── Gemeinsamer State (Subgraph + Hauptgraph) ────────────────────────────
class AnalyseState(TypedDict):
    thema:       str
    roh_info:    str   # Sammler-Ergebnis
    verifiziert: str   # Faktenprüfer-Ergebnis
    bericht:     str   # Finaler Bericht (Hauptgraph)

# ── Subgraph: Informations-Team ──────────────────────────────────────────
def sammler_fn(state: AnalyseState) -> dict:
    """Sammelt 4 Stichpunkte zum Thema."""
    resp = llm.invoke([
        SystemMessage("Sammle 4 prägnante Stichpunkte zu folgendem Thema. Deutsch."),
        HumanMessage(f"Thema: {state['thema']}"),
    ])
    return {"roh_info": resp.content}

def faktenpruefer_fn(state: AnalyseState) -> dict:
    """Prüft die Stichpunkte auf Plausibilität und ergänzt ggf."""
    resp = llm.invoke([
        SystemMessage("Du bist Fakten-Checker. Überprüfe die Stichpunkte: "
                      "Korrigiere Fehler, ergänze wichtige Aspekte. Antworte auf Deutsch."),
        HumanMessage(f"Stichpunkte:\n{state['roh_info']}"),
    ])
    return {"verifiziert": resp.content}

info_builder = StateGraph(AnalyseState)
info_builder.add_node("sammler",       sammler_fn)
info_builder.add_node("faktenpruefer", faktenpruefer_fn)
info_builder.add_edge(START,      "sammler")
info_builder.add_edge("sammler",  "faktenpruefer")
info_builder.add_edge("faktenpruefer", END)

info_subgraph = info_builder.compile()
print("✅ Subgraph: Informations-Team kompiliert")

In [ ]:


# ── Hauptgraph: Subgraph als Node ────────────────────────────────────────
def bericht_fn(state: AnalyseState) -> dict:
    """Schreibt einen Bericht aus den verifizierten Informationen."""
    resp = llm.invoke([
        SystemMessage("Du bist Redakteur. Schreibe einen strukturierten Bericht "
                      "(Einleitung, 3 Punkte, Fazit). Max. 150 Wörter. Deutsch."),
        HumanMessage(f"Verifizierte Informationen:\n{state['verifiziert']}"),
    ])
    return {"bericht": resp.content}

haupt_builder = StateGraph(AnalyseState)
haupt_builder.add_node("informations_team", info_subgraph)  # ← Subgraph als Node
haupt_builder.add_node("bericht_schreiber", bericht_fn)
haupt_builder.add_edge(START,                "informations_team")
haupt_builder.add_edge("informations_team",  "bericht_schreiber")
haupt_builder.add_edge("bericht_schreiber",  END)

hierarchie_graph = haupt_builder.compile()
print("✅ Hierarchischer Hauptgraph kompiliert")

In [ ]:
display(IPImage(info_subgraph.get_graph().draw_mermaid_png()))


In [ ]:
run_cfg_h = {"run_name": "M19-Kap4-Hierarchisch", "tags": ["m19", "hierarchisch"]}

result_h = hierarchie_graph.invoke(
    {"thema": "Erneuerbare Energien und ihre Rolle in der Energiewende",
     "roh_info": "", "verifiziert": "", "bericht": ""},
    config=run_cfg_h
)

mprint("**Rohdaten (Sammler):**\n" + result_h["roh_info"] + "\n---")
mprint("**Verifiziert (Faktenprüfer):**\n" + result_h["verifiziert"] + "\n---")
mprint("**Finaler Bericht:**\n" + result_h["bericht"])

# 5 | Pattern 3: Pipeline
---

Das **Pipeline-Pattern** verbindet Agenten in einer festen, deterministischen Reihenfolge.  
Kein LLM-Routing – der Workflow ist zur Build-Zeit vollständig definiert.

```python
builder.add_edge(START,      "sammler")
builder.add_edge("sammler",  "bewerter")
builder.add_edge("bewerter", "redakteur")
builder.add_edge("redakteur", END)
```

**Stärken der Pipeline:**

| Eigenschaft | Beschreibung |
|-------------|-------------|
| **Deterministisch** | Gleicher Input → gleicher Ablauf |
| **Einfach debuggbar** | Fehler immer in einem bestimmten Schritt |
| **Günstig** | Kein Supervisor-LLM-Aufruf für Routing |
| **Gut skalierbar** | Schritte können parallelisiert werden (M20) |

**Limitation:** Kein dynamisches Routing – immer derselbe Pfad.  
Bei bedingter Logik: Conditional Edges (*Conditional Routing & Tool-Loop*) oder Supervisor (Kapitel 3) verwenden.

> **Typischer Einsatz:** Content-Erzeugung, ETL-Workflows,  
> Datenverarbeitung mit klaren Transformationsschritten.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Pipeline-Pattern Detail</font> </br></p>

diagram = '''
flowchart LR
    START([START]) --> SAM["📰 Sammler\nNews generieren"]
    SAM -->|"news\_items"| BEW["⭐ Bewerter\nRelevanz &amp; Kategorie"]
    BEW -->|"bewertung"| RED["📝 Redakteur\nNewsletter schreiben"]
    RED --> END([END])

    style SAM fill:#2196F3,color:#fff
    style BEW fill:#FF9800,color:#fff
    style RED fill:#4CAF50,color:#fff
'''

mermaid(diagram, width=850)

In [ ]:
# ── State ──────────────────────────────────────────────────────────────────
class NewsletterState(TypedDict):
    thema:      str
    news_items: str   # Sammler-Ergebnis: generierte News-Meldungen
    bewertung:  str   # Bewerter-Ergebnis: Relevanz + Kategorien
    newsletter: str   # Redakteur-Ergebnis: fertiger Newsletter-Abschnitt

# ── Agenten ────────────────────────────────────────────────────────────────
def news_sammler(state: NewsletterState) -> dict:
    """Generiert 3 News-Meldungen zum Thema (simuliert Recherche)."""
    resp = llm.invoke([
        SystemMessage("Generiere 3 kurze, realistische News-Meldungen zu dem Thema. "
                      "Format: Überschrift: Kurzbeschreibung (1 Satz). Deutsch."),
        HumanMessage(f"Thema: {state['thema']}"),
    ])
    return {"news_items": resp.content}

def news_bewerter(state: NewsletterState) -> dict:
    """Bewertet News nach Relevanz und kategorisiert sie."""
    resp = llm.invoke([
        SystemMessage("Bewerte jede Meldung nach Relevanz (1-5 Sterne) und "
                      "kategorisiere sie (Technik/Wirtschaft/Gesellschaft). Deutsch."),
        HumanMessage(f"News-Meldungen:\n{state['news_items']}"),
    ])
    return {"bewertung": resp.content}

def newsletter_redakteur(state: NewsletterState) -> dict:
    """Schreibt einen professionellen Newsletter-Abschnitt."""
    resp = llm.invoke([
        SystemMessage("Schreibe einen professionellen Newsletter-Abschnitt "
                      "basierend auf den bewerteten Meldungen. "
                      "Stil: sachlich, prägnant. Max. 120 Wörter. Deutsch."),
        HumanMessage(f"Bewertete Meldungen:\n{state['bewertung']}"),
    ])
    return {"newsletter": resp.content}

# ── Pipeline-Graph ──────────────────────────────────────────────────────────
pipeline_builder = StateGraph(NewsletterState)
pipeline_builder.add_node("sammler",   news_sammler)
pipeline_builder.add_node("bewerter",  news_bewerter)
pipeline_builder.add_node("redakteur", newsletter_redakteur)

pipeline_builder.add_edge(START,       "sammler")
pipeline_builder.add_edge("sammler",   "bewerter")
pipeline_builder.add_edge("bewerter",  "redakteur")
pipeline_builder.add_edge("redakteur", END)

pipeline_graph = pipeline_builder.compile()
print("✅ Pipeline-Graph kompiliert")

In [ ]:
display(IPImage(pipeline_graph.get_graph().draw_mermaid_png()))

In [ ]:
run_cfg_p = {"run_name": "M19-Kap5-Pipeline", "tags": ["m19", "pipeline"]}

result_p = pipeline_graph.invoke(
    {"thema": "Künstliche Intelligenz in der Medizin",
     "news_items": "", "bewertung": "", "newsletter": ""},
    config=run_cfg_p
)

mprint("**Gesammelte News:**\n" + result_p["news_items"] + "\n---")
mprint("**Bewertung:**\n" + result_p["bewertung"] + "\n---")
mprint("**Newsletter-Abschnitt:**\n" + result_p["newsletter"])

# 6 | Pattern-Vergleich
---

| Kriterium | Supervisor | Hierarchisch | Pipeline |
|-----------|-----------|--------------|----------|
| **Routing** | LLM-gesteuert, dynamisch | Fest + Sub-Teams | Fest, sequentiell |
| **Flexibilität** | ⭐⭐⭐ hoch | ⭐⭐ mittel | ⭐ niedrig |
| **Kosten** | Höher (Supervisor-LLM) | Mittel | Niedriger |
| **Debuggbarkeit** | Schwerer (non-deterministisch) | Gut | Sehr gut |
| **Skalierbarkeit** | Mittel | Hoch | Hoch |
| **Implementierung** | Mittel | Komplex | Einfach |

**Entscheidungshilfe:**

```
Aufgabe klar in feste Schritte aufteilbar?
  → JA:  Pipeline
  → NEIN: Dynamisches Routing nötig?
      → JA:  Supervisor
      → NEIN: Komplexe Teilaufgaben, wiederverwendbar?
          → JA:  Hierarchisch (Subgraphen)
          → NEIN: Pipeline mit Conditional Edges (*Conditional Routing & Tool-Loop*)
```

**Kombinationen sind möglich** – z.B. Supervisor auf oberster Ebene,  
der spezialisierte Subgraphen (Hierarchisch) als Agenten nutzt.  
*Supervisor Pattern* und *Multi-Agent Projekt* vertiefen diese Kombinationen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M19-Multi-Agent-Patterns", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
News-Analyse-System als Multi-Agent
</font></p>

Bauen Sie ein News-Analyse-System, das drei verschiedene Quellen (simuliert) auswertet  
und einen konsolidierten Report erstellt.

**Variante A – Supervisor-Ansatz:**
```python
class NewsAnalyseState(TypedDict):
    messages:   Annotated[list, add_messages]
    thema:      str
    naechster:  str
    # Agenten: technik_agent, wirtschaft_agent, gesellschaft_agent, synthese_agent
    # Supervisor entscheidet Reihenfolge dynamisch
```

**Variante B – Pipeline-Ansatz:**
```python
class NewsReportState(TypedDict):
    thema:       str
    raw_news:    str   # sammler
    kategorien:  str   # kategorisierer  
    zusammenfassung: str  # zusammenfasser
    headline:    str   # headline_generator
```

**Bonus:**
- HITL (aus *Human-in-the-Loop*) einbauen: Mensch genehmigt den Report vor Veröffentlichung
- Checkpointing (aus *Checkpointing & Sessions*): Reports persistent speichern (SqliteSaver)
- Subgraph aus Section 4: Jede "Quelle" ist ein eigenes Informations-Team